# LifeMate — full audio-to-tools validation, no fine-tuning
Qwen3-4B-Instruct-2507 original weights + supplied asr_model/ Zipformer2 ONNX.

- Audio (current request and user history) → Sherpa-ONNX ASR → Planner → Caller → ToolEnv → Summarizer.
- All 100 validation IDs from split.txt; max 4 independent workflows per vLLM batch.
- No training, optimizer, Unsloth installation or trained LoRA loading. Inference rationale is generated by Qwen.
- No audio_text/oracle fallback. ASR cache, predictions, traces and local state-hash/tool-F1 scores are saved.
- ASR uses an isolated CPU runtime; vLLM uses one T4 with FP16 base weights. No per-sample time cutoff.
- Default MODES=("multi",); optional ("single","multi") also evaluates the single-agent global prompt.
- VALID_LIMIT=None runs all 100; VALID_LIMIT=4 is a shorter full-pipeline smoke run.
- This measures the pipeline before global-to-local fine-tuning, not a trained paper result.

Kaggle inputs: cymeu4l0t/zaloai-dataset and private lctrnnguynkhi/lifemate-supplied-zipformer-asr.


In [ ]:
from pathlib import Path
import subprocess, sys, json, hashlib, zipfile, os

REPO_URL = "https://github.com/tnisl/baseline.git"
GIT_REF = "4894d331692293565635454a8c686b7c18314a04"
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"
RUN_TRAINING = False
assert RUN_TRAINING is False,"This notebook is validation-only"
VALID_LIMIT = None  # None: all 100 validation IDs; 4: short pipeline smoke run
MODES = ("multi",)  # Optional comparison: ("single", "multi")
BATCH_SIZE = 4
ASR_THREADS = 4
EVAL_DEADLINE_S = None  # No per-sample time cutoff during quality evaluation
SOURCE = Path("/kaggle/working/paper_source_unsloth_vllm")
assert VALID_LIMIT is None or 4<=VALID_LIMIT<=100
assert set(MODES)<={"single","multi"} and MODES
if not SOURCE.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(SOURCE)], check=True)
subprocess.run(["git", "-C", str(SOURCE), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(SOURCE), "checkout", "--detach", "FETCH_HEAD"], check=True)
SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip()
assert SOURCE_COMMIT == GIT_REF
print("Source:", SOURCE_COMMIT, "Mode: validation_only", "Limit:", VALID_LIMIT)


In [ ]:
INFER_SOURCE='"""Validation-only base-model workflow; no training, labels or adapters in prompts."""\nimport os\nos.environ.setdefault("CUDA_VISIBLE_DEVICES","0")\nos.environ.setdefault("VLLM_USE_V1","0")\nos.environ.setdefault("VLLM_ATTENTION_BACKEND","XFORMERS")\nos.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD","spawn")\nos.environ.setdefault("VLLM_USAGE_STATS_ENABLED","0")\nos.environ.setdefault("DO_NOT_TRACK","1")\nimport json,time,argparse,math\nfrom pathlib import Path\nfrom collections import Counter\nfrom umilife.core import read_jsonl,write_json,dumps,messages,parse_json,allowed_specs,load_catalog,import_toolenv,SYSTEMS\nfrom umilife.data import normalized_row\nfrom umilife.grammar import role_schema\n\ndef rollout(row,text,specs,env,mode,max_calls=10,max_decisions=16):\n    started=time.monotonic();trajectory=[];trace=[];answer=None;reason="decision_limit"\n    def generate(role,plan=None,feedback=None):\n        response=yield {"role":role,"messages":messages(role,row,text,specs,trajectory,plan,feedback)}\n        trace.append({"role":role,**response})\n        if response.get("schema_valid") is False:\n            raise ValueError("Model output violates role JSON schema: "+response.get("schema_error",""))\n        return parse_json(response["raw"])\n    for _ in range(max_decisions):\n        try:\n            role="global" if mode=="single" else "planner"\n            plan=yield from generate(role)\n            trace.append({"role":role,"output":plan})\n            route=plan.get("next")\n            if route=="give_up":reason="give_up";break\n            if route=="summarizer":\n                answer=plan.get("answer") if mode=="single" else (yield from generate("summarizer",plan)).get("answer")\n                reason="complete";break\n            if route!="caller":raise ValueError("invalid planner route")\n            if len(env.get_tool_calls())>=max_calls:reason="call_limit";break\n            action=plan.get("action") if mode=="single" else (yield from generate("caller",plan))\n            for attempt in range(2):\n                if not isinstance(action,dict):raise ValueError("invalid action")\n                name,args=action.get("name"),action.get("arguments")\n                check=env.validate(name,args) if name in specs and isinstance(args,dict) else {"ok":False,"error":"Unknown tool or invalid arguments"}\n                trace.append({"role":"caller","output":action,"validation":check})\n                if check.get("ok"):break\n                if attempt or mode=="single":raise ValueError("invalid tool schema")\n                action=yield from generate("caller",plan,{"action":action,"result":check})\n            observed=env.call(name,args)\n            trajectory.append({"thought":plan.get("thought",""),"action":action,"observation":observed})\n        except (ValueError,KeyError,TypeError) as exc:\n            reason="format_error";trace.append({"error":str(exc)});break\n    return {"id":row["id"],"tool_calls":env.get_tool_calls(),"state_hash":env.get_state_hash()},{\n        "id":row["id"],"reason":reason,"elapsed_s":time.monotonic()-started,\n        "over_deadline":False,"time_limit_enforced":False,"answer":answer,\n        "trace":trace,"trajectory":trajectory,"transcript":text}\n\ndef run_batch(generators,backend):\n    active={}\n    completed={}\n    for index,generator in enumerate(generators):\n        try:active[index]=next(generator)\n        except StopIteration as stop:completed[index]=stop.value\n    while active:\n        good=[];bad=[]\n        for index,request in active.items():\n            try:backend.validate(request);good.append(index)\n            except ValueError as exc:bad.append((index,exc))\n        for index,error in bad:\n            try:active[index]=generators[index].throw(error)\n            except StopIteration as stop:completed[index]=stop.value;del active[index]\n        if not good:continue\n        responses=backend.generate([active[index] for index in good])\n        assert len(responses)==len(good)\n        for index,response in zip(good,responses,strict=True):\n            try:active[index]=generators[index].send(response)\n            except StopIteration as stop:completed[index]=stop.value;del active[index]\n    return [completed[index] for index in range(len(generators))]\n\nclass Backend:\n    def __init__(self,model,revision,adapters=None,max_context=8192):\n        assert adapters is None,"Validation-only baseline must not load trained adapters"\n        import torch\n        from vllm import LLM\n        from transformers import AutoConfig,AutoModelForCausalLM\n        assert torch.cuda.is_available()\n        assert torch.cuda.get_device_capability(0)>=(7,0)\n        self.max_context=max_context\n        self.requests={role:None for role in ("global","planner","caller","summarizer")}\n        self.t4_compatibility={"applied":False,"reason":"Base-only inference; no LoRA kernels"}\n        config=AutoConfig.from_pretrained(model,revision=revision)\n        with torch.device("meta"):meta_model=AutoModelForCausalLM.from_config(config)\n        base_parameters=meta_model.num_parameters();del meta_model\n        self.parameter_report={"base":base_parameters,"comparison_adapters":0,"total_llm":base_parameters}\n        assert base_parameters<5_000_000_000\n        self.llm=LLM(model=model,revision=revision,tokenizer=model,tokenizer_revision=revision,\n            dtype="float16",tensor_parallel_size=1,enable_lora=False,\n            max_model_len=max_context,max_num_seqs=4,max_num_batched_tokens=max_context,\n            gpu_memory_utilization=.90,enforce_eager=True,enable_prefix_caching=False,\n            enable_chunked_prefill=False,seed=42,trust_remote_code=False,\n            generation_config="vllm",disable_log_stats=True)\n        self.tokenizer=self.llm.get_tokenizer()\n        self.max_tokens={role:192 if role=="planner" else 512 for role in self.requests}\n        self.batch_sizes=[];self.roles_seen=Counter();self.schema_violations=0\n    def make_sampling(self,request):\n        from vllm import SamplingParams\n        from vllm.sampling_params import GuidedDecodingParams\n        # Fresh per request: never share mutable grammar state between batch members.\n        return SamplingParams(temperature=0,seed=42,\n            max_tokens=64 if "schema" in request else self.max_tokens[request["role"]],\n            guided_decoding=GuidedDecodingParams(\n                json=request.get("schema",role_schema(request["role"])),backend="lm-format-enforcer"))\n    def validate(self,request):\n        role=request["role"]\n        prompt=self.tokenizer.apply_chat_template(request["messages"],tokenize=False,add_generation_prompt=True)\n        length=len(self.tokenizer.encode(prompt,add_special_tokens=False))\n        if length+(64 if "schema" in request else self.max_tokens[role])>self.max_context:\n            raise ValueError("Context budget exceeded; no evidence is truncated")\n    def generate(self,requests):\n        assert 1<=len(requests)<=4\n        for r in requests:self.validate(r)\n        self.batch_sizes.append(len(requests));self.roles_seen.update(r["role"] for r in requests)\n        prompts=[self.tokenizer.apply_chat_template(r["messages"],tokenize=False,add_generation_prompt=True) for r in requests]\n        results=self.llm.generate(prompts,[self.make_sampling(r) for r in requests],\n            use_tqdm=False)\n        from jsonschema import Draft202012Validator,ValidationError\n        responses=[]\n        for request,result in zip(requests,results,strict=True):\n            response={"raw":result.outputs[0].text,"finish_reason":result.outputs[0].finish_reason,\n                      "stop_reason":result.outputs[0].stop_reason}\n            try:\n                value=parse_json(response["raw"])\n                Draft202012Validator(request.get("schema",role_schema(request["role"]))).validate(value)\n                response["schema_valid"]=True\n            except (ValueError,ValidationError) as exc:\n                response.update(schema_valid=False,schema_error=str(exc)[:300])\n                self.schema_violations+=1\n            responses.append(response)\n        return responses\n    def preflight(self):\n        probes=[]\n        for role in ("global","planner","caller","summarizer"):\n            schema={"type":"object","properties":{"probe":{"type":"string","enum":[role]}},\n                    "required":["probe"],"additionalProperties":False}\n            probes.append({"role":role,"schema":schema,"messages":[\n                {"role":"user","content":"Just say hello, no JSON."}]})\n        for request,result in zip(probes,self.generate(probes),strict=True):\n            assert result["schema_valid"] and parse_json(result["raw"])=={"probe":request["role"]}, (\n                "Engine ignored the request\'s grammar",request["role"],result)\n        requests=[]\n        for role in ("global","planner","caller","summarizer"):\n            requests.append({"role":role,"messages":[{"role":"system","content":SYSTEMS[role]},\n                {"role":"user","content":dumps({"context":{},"request":"Chào bạn","history":[],\n                    "trajectory":[],"tools":[],"plan":{"thought":"Không có thao tác cần thực hiện.","next":"give_up"}})}]})\n        results=self.generate(requests)\n        for request,result in zip(requests,results,strict=True):\n            assert result["schema_valid"],result\n            value=parse_json(result["raw"])\n            if request["role"] in ("global","planner"):\n                assert value["next"] in ("caller","summarizer","give_up")\n            elif request["role"]=="caller":\n                assert isinstance(value["name"],str) and isinstance(value["arguments"],dict)\n            else:assert isinstance(value["answer"],str)\n        assert self.batch_sizes[-1]==4\n        return {"real_vllm_batch_size":4,"tested_prompt_roles":list(self.requests),"json_contracts":"pass","adversarial_schema_probe":"pass"}\n\ndef main():\n    p=argparse.ArgumentParser()\n    for name in ("data-root","work","model","revision"):p.add_argument("--"+name,required=True)\n    p.add_argument("--limit",type=int)\n    p.add_argument("--modes",nargs="+",choices=("single","multi"),default=["multi"])\n    p.add_argument("--max-calls",type=int,default=10)\n    p.add_argument("--max-decisions",type=int,default=16)\n    a=p.parse_args();root=Path(a.data_root);work=Path(a.work);data=work/"prepared"\n    manifest=json.loads((data/"manifest.json").read_text())\n    valid={key for key,value in manifest["split"].items() if value=="validation"}\n    rows=[r for r in read_jsonl(root/"train/train.jsonl") if r["id"] in valid]\n    assert len(rows)==100\n    if a.limit:rows=rows[:a.limit]\n    backend=Backend(a.model,a.revision)\n    preflight=backend.preflight();write_json(work/"vllm_preflight.json",preflight)\n    print(dumps({"vllm_preflight":preflight}),flush=True)\n    transcripts=json.loads((work/"asr_transcripts.json").read_text())\n    asr_report=json.loads((work/"asr_report.json").read_text())\n    assert asr_report["validation_samples"]==len(rows) and not asr_report["oracle_fallback"]\n    for row in rows:\n        required=[row["audio"]]+[t["audio"] for t in row.get("history",[]) if t["role"]=="user"]\n        assert all(name in transcripts for name in required),"Missing current/history ASR transcript"\n    catalog=load_catalog(root,"train");ToolEnv=import_toolenv(root,"train")\n    from umilife.evaluate import evaluate\n    references=json.loads((data/"references.json").read_text())\n    summary={}\n    for mode in a.modes:\n        name=f"validation_{mode}_base_asr_vllm_batch4"\n        predictions=[];traces=[]\n        with (work/f"{name}.jsonl").open("w",encoding="utf-8") as pf,(work/f"{name}.traces.jsonl").open("w",encoding="utf-8") as tf:\n            for start in range(0,len(rows),4):\n                group=rows[start:start+4]\n                # One independent ToolEnv per sample, never shared across batch members.\n                envs=[ToolEnv("train",raw["id"],root=root) for raw in group]\n                safe_rows=[normalized_row(raw,transcript=transcripts) for raw in group]\n                gens=[rollout(row,transcripts[raw["audio"]],allowed_specs(row,catalog),env,mode,\n                              a.max_calls,a.max_decisions)\n                      for raw,row,env in zip(group,safe_rows,envs,strict=True)]\n                results=run_batch(gens,backend)\n                for prediction,trace in results:\n                    predictions.append(prediction);traces.append(trace)\n                    pf.write(dumps(prediction)+"\\n");pf.flush()\n                    tf.write(dumps(trace)+"\\n");tf.flush()\n                print(dumps({"mode":mode,"completed":len(predictions),"total":len(rows),\n                    "batch":len(group),"reasons":dict(Counter(t["reason"] for _,t in results))}),flush=True)\n        assert len(predictions)==len(rows) and {r["id"] for r in predictions}=={r["id"] for r in rows}\n        report=evaluate(predictions,references,catalog,traces)\n        report.update(time_limit_enforced=False,inference_engine="vllm",batch_size=4,\n                      metric_note="Quality-only local approximation; no time penalty.")\n        write_json(work/f"{name}.metrics.json",report)\n        summary[mode]={k:v for k,v in report.items() if k!="per_sample"}\n        print(dumps({"mode":mode,"metrics":summary[mode]}),flush=True)\n    assert backend.parameter_report["total_llm"]+asr_report["initializer_elements"]<6_000_000_000\n    runtime={"engine":"vllm","batch_size":4,"actual_batch_sizes":dict(Counter(backend.batch_sizes)),\n        "roles_seen":dict(backend.roles_seen),"dtype":"float16","base_quantization":None,\n        "training_quantization":None,"finetuned":False,"adapter_loaded":False,"execution_mode":"validation_only","parameters":backend.parameter_report,\n        "schema_enforced":True,"schema_violations":backend.schema_violations,\n        "t4_compatibility":backend.t4_compatibility,\n        "max_context":backend.max_context,"asr":True,"asr_model":"supplied_zipformer2","asr_report":asr_report,"time_limit_enforced":False,\n        "samples_per_mode":len(rows),"metrics":summary}\n    assert max(backend.batch_sizes)==4\n    write_json(work/"vllm_runtime.json",runtime)\nif __name__=="__main__":main()\n'
ASR_SOURCE='"""Recognize current/history audio using only the supplied Zipformer2 ONNX files."""\nimport argparse,hashlib,json,math,time\nfrom pathlib import Path\nimport numpy as np\nimport soundfile as sf\nfrom scipy.signal import resample_poly\nimport sherpa_onnx\n\ndef digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef transcribe(root,work,model,limit=None,threads=4):\n    root=Path(root);work=Path(work);model=Path(model)\n    manifest=json.loads((model/\'model_manifest.json\').read_text())\n    assert manifest[\'model_type\']==\'zipformer2\' and not manifest[\'streaming\']\n    for name,info in manifest[\'files\'].items():assert digest(model/name)==info[\'sha256\'],f\'Wrong ASR file: {name}\'\n    fingerprint=digest(model/\'model_manifest.json\')\n    valid=set((work/\'split.txt\').read_text().splitlines());assert len(valid)==100\n    rows=[json.loads(line) for line in (root/\'train/train.jsonl\').open()]\n    selected=[r for r in rows if r[\'id\'] in valid];assert len(selected)==100\n    if limit:selected=selected[:limit]\n    audio=[]\n    for row in selected:\n        audio.append(row[\'audio\'])\n        audio.extend(t[\'audio\'] for t in row.get(\'history\',[]) if t[\'role\']==\'user\')\n    audio=list(dict.fromkeys(audio))\n    config={\'model_manifest_sha256\':fingerprint,\'split_sha256\':digest(work/\'split.txt\'),\n        \'selected_ids\':[r[\'id\'] for r in selected],\'sample_rate\':16000,\'decoding_method\':\'greedy_search\',\n        \'provider\':\'cpu\',\'sherpa_onnx\':sherpa_onnx.__version__,\'source\':\'supplied_asr_model\'}\n    config_path=work/\'asr_config.json\'\n    if config_path.exists():assert json.loads(config_path.read_text())==config,\'Stale ASR cache configuration\'\n    else:config_path.write_text(json.dumps(config,indent=2))\n    cache_path=work/\'asr_transcripts.jsonl\';cache={}\n    if cache_path.exists():\n        for line in cache_path.read_text().splitlines():\n            record=json.loads(line);assert record[\'model_manifest_sha256\']==fingerprint\n            cache[record[\'audio\']]=record\n    recognizer=sherpa_onnx.OfflineRecognizer.from_transducer(\n        encoder=str(model/\'encoder.onnx\'),decoder=str(model/\'decoder.onnx\'),joiner=str(model/\'joiner.onnx\'),\n        tokens=str(model/\'tokens.txt\'),num_threads=threads,sample_rate=16000,feature_dim=80,\n        decoding_method=\'greedy_search\',provider=\'cpu\')\n    started=time.monotonic()\n    with cache_path.open(\'a\',encoding=\'utf-8\') as output:\n        for index,name in enumerate(audio,1):\n            path=(root/\'train\'/name).resolve();assert path.is_relative_to((root/\'train\').resolve())\n            audio_sha=digest(path)\n            if name in cache:\n                assert cache[name][\'audio_sha256\']==audio_sha,\'Audio changed after transcription\'\n                continue\n            tick=time.monotonic()\n            try:\n                samples,sample_rate=sf.read(path,dtype=\'float32\',always_2d=True)\n                samples=samples.mean(axis=1)\n                assert len(samples)>0 and np.isfinite(samples).all(),\'Empty or non-finite waveform\'\n                duration=len(samples)/sample_rate\n                if sample_rate!=16000:\n                    factor=math.gcd(sample_rate,16000)\n                    samples=resample_poly(samples,16000//factor,sample_rate//factor)\n                stream=recognizer.create_stream()\n                stream.accept_waveform(16000,np.ascontiguousarray(samples,dtype=np.float32))\n                recognizer.decode_stream(stream)\n                record={\'audio\':name,\'text\':stream.result.text.strip(),\'audio_sha256\':audio_sha,\n                    \'model_manifest_sha256\':fingerprint,\'elapsed_s\':time.monotonic()-tick,\'duration_s\':duration,\n                    \'source\':\'supplied_asr_model\',\'provider\':\'cpu\'}\n            except Exception as exc:\n                (work/\'asr_error.json\').write_text(json.dumps({\'audio\':name,\'error\':str(exc)}))\n                raise\n            output.write(json.dumps(record,ensure_ascii=False)+\'\\n\');output.flush();cache[name]=record\n            print(json.dumps({\'stage\':\'asr\',\'completed\':index,\'total_audio\':len(audio),\n                \'validation_samples\':len(selected),\'audio\':name,\'elapsed_s\':record[\'elapsed_s\']},ensure_ascii=False),flush=True)\n    assert set(audio)<=cache.keys()\n    transcripts={name:cache[name][\'text\'] for name in audio}\n    (work/\'asr_transcripts.json\').write_text(json.dumps(transcripts,ensure_ascii=False,indent=2))\n    report={\'status\':\'complete\',\'validation_samples\':len(selected),\'unique_audio\':len(audio),\n        \'current_and_history_audio\':True,\'oracle_fallback\':False,\'training\':False,\n        \'empty_transcripts\':sum(not value.strip() for value in transcripts.values()),\n        \'wall_s_this_run\':time.monotonic()-started,\n        \'audio_duration_s\':sum(cache[name][\'duration_s\'] for name in audio),\n        \'initializer_elements\':sum(info.get(\'initializer_elements\',0) for info in manifest[\'files\'].values()),\n        \'model_manifest_sha256\':fingerprint,\'provider\':\'cpu\',\'model_type\':\'zipformer2\'}\n    (work/\'asr_report.json\').write_text(json.dumps(report,indent=2))\n    print(\'ASR_COMPLETE\',json.dumps(report),flush=True)\n    return report\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser()\n    for name in (\'data-root\',\'work\',\'model-dir\'):p.add_argument(\'--\'+name,required=True)\n    p.add_argument(\'--limit\',type=int);p.add_argument(\'--threads\',type=int,default=4)\n    a=p.parse_args();transcribe(a.data_root,a.work,a.model_dir,a.limit,a.threads)\n'
INFER_REQUIREMENTS='--extra-index-url https://download.pytorch.org/whl/cu128\naccelerate==1.15.0\nagent-detector==2.0.0\naiohappyeyeballs==2.7.1\naiohttp==3.14.5\naiosignal==1.4.0\nairportsdata==20261030\nannotated-doc==0.0.5\nannotated-types==0.8.0\nanyio==4.16.0\nastor==0.8.1\nattrs==26.1.0\nblake3==1.0.11\ncachetools==7.2.8\ncertifi==2026.10.23\ncharset-normalizer==3.5.2\nclick==8.5.0\ncloudpickle==3.1.2\ncompressed-tensors==0.10.2\ncuda-pathfinder==1.8.5\ncupy-cuda12x==14.2.0\ndepyf==0.18.0\ndetect-installer==0.2.1\ndill==0.4.1\ndiskcache==5.6.3\ndistro==1.9.0\ndnspython==2.9.1\neinops==0.8.2\nemail-validator==2.3.0\nfastapi==0.162.0\nfastapi-cli==0.0.32\nfastapi-cloud-cli==0.26.0\nfastar==0.12.0\nfilelock==4.2.7\nfrozenlist==1.8.0\nfsspec==2026.9.0\ngguf==0.19.0\ngoogleapis-common-protos==1.76.0\nh11==0.16.0\nhf-xet==1.8.0\nhttpcore==1.0.9\nhttpcore2==2.16.0\nhttptools==0.9.0\nhttpx==0.28.1\nhttpx2==2.16.0\nhuggingface-hub==0.35.3\nidna==3.20\ninteregular==0.3.3\njinja2==3.1.6\njiter==0.18.0\njsonschema==4.25.1\njsonschema-specifications==2025.9.1\nlark==1.2.2\nllguidance==0.7.30\nllvmlite==0.44.0\nlm-format-enforcer==0.10.12\nmarkdown-it-py==4.2.0\nmarkupsafe==3.0.4\nmdurl==0.1.2\nmistral-common==1.13.1\nmpmath==1.3.0\nmsgpack==1.2.4\nmsgspec==0.22.0\nmultidict==7.2.1\nnest-asyncio==1.6.0\nnetworkx==3.7\nninja==1.13.2\nnumba==0.61.2\nnumpy==2.2.6\nnvidia-cublas-cu12==12.8.3.14\nnvidia-cuda-cupti-cu12==12.8.57\nnvidia-cuda-nvrtc-cu12==12.8.61\nnvidia-cuda-runtime-cu12==12.8.57\nnvidia-cudnn-cu12==9.7.1.26\nnvidia-cufft-cu12==11.3.3.41\nnvidia-cufile-cu12==1.13.0.11\nnvidia-curand-cu12==10.3.9.55\nnvidia-cusolver-cu12==11.7.2.55\nnvidia-cusparse-cu12==12.5.7.53\nnvidia-cusparselt-cu12==0.6.3\nnvidia-nccl-cu12==2.26.2\nnvidia-nvjitlink-cu12==12.8.61\nnvidia-nvtx-cu12==12.8.55\nopenai==1.90.0\nopencv-python-headless==5.0.0.93\nopentelemetry-api==1.45.2\nopentelemetry-exporter-http-transport==0.66b2\nopentelemetry-exporter-otlp-common==0.66b2\nopentelemetry-exporter-otlp-proto-common==1.45.2\nopentelemetry-exporter-otlp-proto-http==1.45.2\nopentelemetry-proto==1.45.2\nopentelemetry-sdk==1.45.2\nopentelemetry-semantic-conventions==0.66b2\noutlines==0.1.11\noutlines-core==0.1.26\npackaging==26.3\npartial-json-parser==0.2.1.1.post7\npeft==0.17.1\npillow==13.0.0\nprometheus-client==0.26.0\nprometheus-fastapi-instrumentator==8.1.0\npropcache==0.5.4\nprotobuf==7.36.2\npsutil==7.2.2\npy-cpuinfo==9.0.0\npybase64==1.5.1\npycountry==26.2.16\npydantic==2.14.2\npydantic-core==2.50.2\npydantic-extra-types==2.11.1\npydantic-settings==2.16.2\npygments==2.21.0\npython-dotenv==1.2.4\npython-json-logger==4.2.0\npython-multipart==0.0.33\npyyaml==6.0.3\npyzmq==27.2.0\nray==2.59.0\nreferencing==0.37.0\nregex==2026.9.29\nrequests==2.34.2\nrich==15.0.0\nrich-toolkit==0.20.6\nrignore==0.8.1\nrpds-py==2026.9.1\nsafetensors==0.8.0\nscipy==1.18.1\nsentencepiece==0.2.2\nsentry-sdk==2.73.0\nsetuptools==79.0.1\nshellingham==1.5.4\nsix==1.17.0\nsniffio==1.3.1\nstarlette==1.10.0\nsympy==1.14.0\ntiktoken==0.14.0\ntokenizers==0.21.4\ntorch==2.7.0+cu128\ntorchaudio==2.7.0+cu128\ntorchvision==0.22.0+cu128\ntqdm==4.70.1\ntransformers==4.53.3\ntriton==3.3.0\ntruststore==0.10.5\ntyper==0.27.3\ntyping-extensions==4.16.0\ntyping-inspection==0.4.4\nurllib3==2.8.0\nuvicorn==0.54.0\nuvloop==0.23.0\nvllm==0.9.2\nwatchfiles==1.3.0\nwebsockets==17.4\nwrapt==1.17.3\nxformers==0.0.30\nxgrammar==0.1.19\nyarl==1.25.1\n'
ASR_REQUIREMENTS='cffi==2.1.1\nnumpy==2.2.6\npycparser==3.12\nscipy==1.16.2\nsherpa-onnx==2.0.1\nsherpa-onnx-core==2.0.1\nsoundfile==0.13.1\n'
RUNTIME_SHA='777fb273bba3e767e1c2c28ba0ba1294545991b5b83dcc815427e5f8de0b0ea1'
assert hashlib.sha256((INFER_SOURCE+ASR_SOURCE+INFER_REQUIREMENTS+ASR_REQUIREMENTS).encode()).hexdigest()==RUNTIME_SHA
(SOURCE/"umilife/infer_base_vllm.py").write_text(INFER_SOURCE)
(SOURCE/"umilife/transcribe_supplied.py").write_text(ASR_SOURCE)
LOCK_FILE=Path("/tmp/lifemate_validation_requirements.txt");LOCK_FILE.write_text(INFER_REQUIREMENTS)
ASR_LOCK_FILE=Path("/tmp/lifemate_asr_requirements.txt");ASR_LOCK_FILE.write_text(ASR_REQUIREMENTS)


In [ ]:
# Install only the previously tested vLLM inference stack; no training environment.
subprocess.run([sys.executable,"-m","pip","install","--quiet","uv==0.9.10"],check=True)
UV=[sys.executable,"-m","uv"];os.environ["UV_NATIVE_TLS"]="true"
RUNTIME=Path("/tmp/lifemate_vllm092_env")
if not (RUNTIME/"bin/python").exists():
    subprocess.run([*UV,"venv","--python","3.12","--python-preference","only-managed","--seed",str(RUNTIME)],check=True)
PYTHON=str(RUNTIME/"bin/python")
subprocess.run([*UV,"pip","sync","--index-strategy","unsafe-best-match","--python",PYTHON,str(LOCK_FILE)],check=True)
subprocess.run([*UV,"pip","check","--python",PYTHON],check=True)
purelib=Path(subprocess.check_output([PYTHON,"-I","-c","import sysconfig;print(sysconfig.get_path('purelib'))"],text=True).strip())
RUN_ENV=os.environ.copy();RUN_ENV.pop("PYTHONPATH",None);RUN_ENV.pop("PYTHONHOME",None)
RUN_ENV.update(CUDA_VISIBLE_DEVICES="0",TOKENIZERS_PARALLELISM="false",PYTHONNOUSERSITE="1",
    VLLM_USE_V1="0",VLLM_ATTENTION_BACKEND="XFORMERS",VLLM_WORKER_MULTIPROC_METHOD="spawn",
    VLLM_USAGE_STATS_ENABLED="0",DO_NOT_TRACK="1",HF_HOME="/tmp/lifemate_hf_cache")
RUN_ENV["LD_LIBRARY_PATH"]=":".join([str(p) for p in sorted((purelib/"nvidia").glob("*/lib"))]+[os.environ.get("LD_LIBRARY_PATH","")])
subprocess.run([PYTHON,"-c","import vllm,torch;assert vllm.__version__=='0.9.2';assert torch.cuda.is_available();print('GPU:',torch.cuda.get_device_name(0),'vLLM:',vllm.__version__)"],env=RUN_ENV,check=True)

# ASR has no Torch/CUDA dependencies and cannot train either model.
ASR_RUNTIME=Path("/tmp/lifemate_supplied_asr_env")
if not (ASR_RUNTIME/"bin/python").exists():
    subprocess.run([*UV,"venv","--python","3.12","--python-preference","only-managed","--seed",str(ASR_RUNTIME)],check=True)
ASR_PYTHON=str(ASR_RUNTIME/"bin/python")
subprocess.run([*UV,"pip","sync","--python",ASR_PYTHON,str(ASR_LOCK_FILE)],check=True)
subprocess.run([*UV,"pip","check","--python",ASR_PYTHON],check=True)
ASR_ENV=os.environ.copy();ASR_ENV.pop("PYTHONPATH",None);ASR_ENV.pop("PYTHONHOME",None)
ASR_ENV.update(PYTHONNOUSERSITE="1",OMP_NUM_THREADS=str(ASR_THREADS))


In [ ]:
sys.path.insert(0, str(SOURCE))
from umilife.paths import find_data_root
DATA_ROOT, archive_path = find_data_root("/kaggle/input")
if DATA_ROOT is None:
    if archive_path is None:
        raise FileNotFoundError("Attach cymeu4l0t/zaloai-dataset.")
    destination = Path("/tmp/competition_data_codex")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            if not (destination/member.filename).resolve().is_relative_to(destination.resolve()):
                raise ValueError("Unsafe archive member")
        archive.extractall(destination)
    DATA_ROOT, _ = find_data_root(destination)
assert DATA_ROOT is not None, "Competition train manifests not found"
TRAIN_SHA = "24c42c79fbe8db932fe9817d041c0d0ca422ac8177b413cc1be0cda1fd8941a8"
assert hashlib.sha256((DATA_ROOT/"train/train.jsonl").read_bytes()).hexdigest() == TRAIN_SHA

ASR_MODEL_DIR=next((p for p in [Path("/kaggle/input/lifemate-supplied-zipformer-asr"),
    Path("/kaggle/input/datasets/lctrnnguynkhi/lifemate-supplied-zipformer-asr")]
    if (p/"model_manifest.json").is_file()),None)
assert ASR_MODEL_DIR is not None,"Attach lctrnnguynkhi/lifemate-supplied-zipformer-asr"
ASR_MODEL_SHA='c1077ee90c0933d76329af5ac4f85eabd5c177dcf94c5bf5adb004dcf5051c9b'
assert hashlib.sha256((ASR_MODEL_DIR/"model_manifest.json").read_bytes()).hexdigest()==ASR_MODEL_SHA


In [ ]:
# Exact bytes read from the user's split.txt; no random/grouped split is used.
VALIDATION_SPLIT_TEXT = 'train_1474\ntrain_0255\ntrain_0063\ntrain_1715\ntrain_0618\ntrain_0538\ntrain_0486\ntrain_0313\ntrain_1703\ntrain_0236\ntrain_1560\ntrain_1713\ntrain_1249\ntrain_0203\ntrain_1356\ntrain_0959\ntrain_0080\ntrain_0075\ntrain_0216\ntrain_0476\ntrain_0508\ntrain_1155\ntrain_1379\ntrain_0066\ntrain_1289\ntrain_0435\ntrain_1653\ntrain_1497\ntrain_1617\ntrain_0953\ntrain_0480\ntrain_1021\ntrain_1353\ntrain_0624\ntrain_0015\ntrain_0354\ntrain_1608\ntrain_0960\ntrain_0779\ntrain_0346\ntrain_0469\ntrain_0772\ntrain_0214\ntrain_0864\ntrain_0225\ntrain_0819\ntrain_0787\ntrain_1384\ntrain_0596\ntrain_0106\ntrain_1623\ntrain_0923\ntrain_1300\ntrain_1486\ntrain_1352\ntrain_1355\ntrain_1014\ntrain_1380\ntrain_1542\ntrain_1295\ntrain_0943\ntrain_1070\ntrain_1705\ntrain_1448\ntrain_0958\ntrain_1081\ntrain_1137\ntrain_1248\ntrain_1478\ntrain_1305\ntrain_0017\ntrain_0977\ntrain_0933\ntrain_1016\ntrain_1128\ntrain_1460\ntrain_1124\ntrain_1658\ntrain_1265\ntrain_1491\ntrain_1432\ntrain_1278\ntrain_1405\ntrain_1546\ntrain_0867\ntrain_1117\ntrain_1654\ntrain_0505\ntrain_1118\ntrain_0572\ntrain_1148\ntrain_1433\ntrain_1556\ntrain_1557\ntrain_1615\ntrain_1317\ntrain_1215\ntrain_1346\ntrain_1440\ntrain_0536\n'
SPLIT_SHA = "631f8cfab61ddde833477f743cf1a65a9012526e0178e97046a36ee864e3d505"
assert hashlib.sha256(VALIDATION_SPLIT_TEXT.encode()).hexdigest() == SPLIT_SHA
VALID_IDS = VALIDATION_SPLIT_TEXT.splitlines()
assert len(VALID_IDS) == len(set(VALID_IDS)) == 100
WORK=Path("/kaggle/working/paper_validation_base_asr_v1")/('all100' if VALID_LIMIT is None else f'smoke{VALID_LIMIT}')
WORK.mkdir(parents=True,exist_ok=True)
DATA=WORK/"prepared";DATA.mkdir(exist_ok=True)
SPLIT_FILE=WORK/"split.txt";SPLIT_FILE.write_text(VALIDATION_SPLIT_TEXT,encoding="utf-8")
RUN_CONFIG={"mode":"validation_only","training":False,"finetuned":False,"adapters":False,
    "model":MODEL,"revision":MODEL_REVISION,"validation_split_sha256":SPLIT_SHA,
    "train_manifest_sha256":TRAIN_SHA,"runtime_sha256":RUNTIME_SHA,"limit":VALID_LIMIT,
    "modes":list(MODES),"batch_size":4,"asr":True,"asr_model_sha256":ASR_MODEL_SHA,"time_limit_enforced":False}
config_path=WORK/"run_config.json"
if config_path.exists():assert json.loads(config_path.read_text())==RUN_CONFIG,"Existing output uses different settings"
else:config_path.write_text(json.dumps(RUN_CONFIG,indent=2))
print("Validation-only output:",WORK)


In [ ]:
# Replay validation gold in a separate scorer-only ToolEnv; rollout environments are created fresh.
PREPARE_CODE='\nimport sys,json\nfrom pathlib import Path\nfrom umilife.core import read_jsonl,write_json,import_toolenv\nroot,work=map(Path,sys.argv[1:]);data=work/"prepared"\nids=(work/"split.txt").read_text().splitlines();valid=set(ids)\nassert len(ids)==len(valid)==100\nrows=read_jsonl(root/"train/train.jsonl")\nassert len(rows)==1545 and valid<={r["id"] for r in rows}\nToolEnv=import_toolenv(root,"train");references=[]\nfor row in rows:\n    if row["id"] not in valid:continue\n    env=ToolEnv("train",row["id"],root=root)\n    gold=row["label"]["tool_calls"];results=row["label"]["tool_results"]\n    assert len(gold)==len(results)\n    for action,expected in zip(gold,results,strict=True):\n        assert env.validate(action["name"],action["arguments"]).get("ok")\n        assert env.call(action["name"],action["arguments"])==expected\n    references.append({"id":row["id"],"tool_calls":gold,"state_hash":env.get_state_hash()})\nassert {r["id"] for r in references}==valid\nwrite_json(data/"references.json",references)\nwrite_json(data/"manifest.json",{"split":{i:"validation" for i in ids},"validation_episodes":100,\n    "execution_mode":"validation_only","training":False,"gold_usage":"scorer only; not model prompts"})\nprint(json.dumps({"validation_episodes":100,"training":False,"references_verified":len(references)}),flush=True)\n'
subprocess.run([PYTHON,"-u","-c",PREPARE_CODE,str(DATA_ROOT),str(WORK)],cwd=SOURCE,env=RUN_ENV,check=True)


In [ ]:
args=[ASR_PYTHON,"-u","-m","umilife.transcribe_supplied","--data-root",str(DATA_ROOT),
      "--work",str(WORK),"--model-dir",str(ASR_MODEL_DIR),"--threads",str(ASR_THREADS)]
if VALID_LIMIT is not None:args += ["--limit",str(VALID_LIMIT)]
(WORK/"status.json").write_text(json.dumps({"status":"transcribing_audio","training":False}))
subprocess.run(args,cwd=SOURCE,env=ASR_ENV,check=True)
asr_report=json.loads((WORK/"asr_report.json").read_text())
assert asr_report["validation_samples"]==(100 if VALID_LIMIT is None else VALID_LIMIT)
assert not asr_report["oracle_fallback"]
print(asr_report)


In [ ]:
args=[PYTHON,"-u","-m","umilife.infer_base_vllm","--data-root",str(DATA_ROOT),
      "--work",str(WORK),"--model",MODEL,"--revision",MODEL_REVISION,"--modes",*MODES]
if VALID_LIMIT is not None:args += ["--limit",str(VALID_LIMIT)]
(WORK/"status.json").write_text(json.dumps({"status":"validating","training":False}))
try:subprocess.run(args,cwd=SOURCE,env=RUN_ENV,check=True)
except Exception as exc:
    (WORK/"status.json").write_text(json.dumps({"status":"validation_failed","error":str(exc),"training":False}));raise
runtime=json.loads((WORK/"vllm_runtime.json").read_text())
assert runtime["execution_mode"]=="validation_only" and not runtime["finetuned"] and not runtime["adapter_loaded"]
assert runtime["samples_per_mode"]==(100 if VALID_LIMIT is None else VALID_LIMIT)
assert runtime["asr"] and not runtime["asr_report"]["oracle_fallback"]
assert runtime["schema_enforced"] and runtime["actual_batch_sizes"].get("4",0)>0
report={"status":"complete","config":RUN_CONFIG,"runtime":runtime}
(WORK/"validation_only_report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))
(WORK/"status.json").write_text(json.dumps({"status":"complete","training":False}))
for mode,metrics in runtime["metrics"].items():print(mode,json.dumps(metrics,ensure_ascii=False,indent=2))
print("Artifacts:",WORK)
